# Lyden 100M — real-data training (M10) on Kaggle

Config: `configs/100m-kaggle.yaml` — vocab 32,768 · dim 512 · 16 layers (mmma) · d_state 128 · seq 1024 · batch 16 × grad_accum 2 = 32,768 tok/step · 7,600 steps ≈ 249M tokens · AMP fp16 (T4 tensor cores) · per-block activation checkpointing.

Run policy: **Save Version → Save & Run All (Commit)** with GPU T4x2 (or P100) and Internet ON.

- Evolution: pip (cached wheels) → clone repo → restore data/ckpts from own previous output → train with auto-resume → eval.
- Commit sessions start with an empty `/kaggle/working`. To continue training across 12 h sessions:
  1. After a run finishes (or times out), open the notebook → **Add Input → Your Notebooks → this notebook → latest version** (its Output carries `Lyden/runs/100m/ckpts`, `lyden-data`, and `wheels`).
  2. Then Save Version → Save & Run All again: cell 3 installs wheels instantly from cache, cell 4 restores data + checkpoints, training resumes from the last ckpt.
- First session only: mamba-ssm compiles from source (~45 min on 4 vCPU, torch 2.11/cp313 has no prebuilt wheels). Built wheels are cached to `/kaggle/working/wheels` so later sessions skip the compile.
- Environment: pin an environment version (don't rely on "always latest") and File → Save so the image id is stored in the notebook — the wheel cache is only valid for the same image.
- If fp16 is unstable or b16 OOMs, fall back inside cell 6: `--config configs/100m-kaggle-fallback.yaml` (batch 8 × accum 4) — same tokens/step and schedule.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
import sys
print(sys.version)

In [ ]:
# Install mamba-ssm + causal-conv1d, fastest source first:
# 1. wheels cached in this notebook's own previous output (attached as Input)
# 2. prebuilt release wheels matched to this runtime's torch/CUDA/cpXX
# 3. source build with --no-build-isolation (pip's isolated build env has no
#    torch, hence the classic crash) — built wheels land in
#    /kaggle/working/wheels so future sessions never compile again.
import glob, os, subprocess, sys, urllib.request, torch

TV = '.'.join(torch.__version__.split('+')[0].split('.')[:2])   # e.g. '2.8'
CU = torch.version.cuda.split('.')[0]                          # '12' or '13'
CP = f'cp{sys.version_info[0]}{sys.version_info[1]}'
print('runtime:', torch.__version__, 'cuda', CU, CP)

JOBS = [
    ('causal_conv1d', 'causal-conv1d==1.7.0',
     f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/'
     f'causal_conv1d-1.7.0+cu{CU}torch{TV}cxx11abiTRUE-{CP}-{CP}-linux_x86_64.whl'),
    ('mamba_ssm', 'mamba-ssm==2.3.2.post1',
     f'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/'
     f'mamba_ssm-2.3.2.post1%2Bcu{CU}torch{TV}cxx11abiTRUE-{CP}-{CP}-linux_x86_64.whl'),
]
for name, pkg, url in JOBS:
    cached = sorted(glob.glob(f'/kaggle/input/*/wheels/{name}-*.whl'))
    if cached:
        print(f'>> pip install (own-output cache) {cached[0]}')
        subprocess.check_call(['pip', 'install', '-q', cached[0]])
        continue
    ok = True
    try:
        urllib.request.urlopen(urllib.request.Request(url, method='HEAD'), timeout=20)
    except Exception as e:
        ok = False
        print(f'[wheel miss] {pkg}: {e}')
    if ok:
        subprocess.check_call(['pip', 'install', '-q', url])
    else:
        os.makedirs('/kaggle/working/wheels', exist_ok=True)
        print(f'>> pip wheel --no-build-isolation {pkg} (CUDA compile, up to ~45 min)')
        subprocess.check_call(['pip', 'wheel', '--no-build-isolation', '-q',
                               '-w', '/kaggle/working/wheels', pkg])
        wheel = sorted(glob.glob(f'/kaggle/working/wheels/{name}-*.whl'))[0]
        subprocess.check_call(['pip', 'install', '-q', wheel])

from mamba_ssm.modules.mamba2 import Mamba2
blk = Mamba2(d_model=512, d_state=128, d_conv=4, expand=2, layer_idx=0, use_mem_eff_path=False).cuda()
with torch.autocast('cuda', dtype=torch.float16):   # match training semantics
    y = blk(torch.randn(2, 256, 512, device='cuda'))
print('mamba-ssm OK:', y.shape, y.dtype, torch.__version__)

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /kaggle/working/Lyden/.git ]; then
  git clone https://github.com/mrrchandra/Lyden.git /kaggle/working/Lyden
else
  cd /kaggle/working/Lyden && git pull --ff-only || echo '(offline keep)'
fi
# Restore previous session's data + checkpoints. Requires this notebook's own
# latest version output attached as Input (Add Input -> Your Notebooks -> ...).
for prev in /kaggle/input/*/; do
  [ -d "$prev/Lyden" ] || continue
  if [ -d "$prev/Lyden/runs/100m/ckpts" ]; then
    mkdir -p /kaggle/working/Lyden/runs/100m/ckpts
    cp -n "$prev"/Lyden/runs/100m/ckpts/*.pt /kaggle/working/Lyden/runs/100m/ckpts/ 2>/dev/null || true
    echo "[resume] checkpoints restored from $prev"
  fi
  if [ -f "$prev/lyden-data/shards/train/manifest.json" ] && [ ! -f /kaggle/working/lyden-data/shards/train/manifest.json ]; then
    cp -r "$prev/lyden-data" /kaggle/working/lyden-data
    echo "[resume] data restored from $prev"
  fi
done
cd /kaggle/working/Lyden
python - <<'PYEOF'
import sys
sys.path.insert(0, 'src')
from lyden.config import load_config
c = load_config('configs/100m-kaggle.yaml')
t = c.training
print('config ok:', t.seq_len * t.batch_size * t.grad_accum, 'tok/step;', t.total_steps, 'steps;', t.amp)
PYEOF

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/Lyden
if [ -f /kaggle/working/lyden-data/shards/train/manifest.json ]; then
  echo '[data] present, skipping preparation'
else
  python scripts/prepare_data.py \
      --dataset tinystories \
      --data-dir /kaggle/working/lyden-data \
      --tokenizer-sample 30000 \
      --max-train-tokens 200000000
fi
du -sh /kaggle/working/lyden-data

In [ ]:
%%bash
# Training with auto-resume; reruns resume from the latest ckpt under runs/100m/ckpts.
# If b16 OOMs or fp16 is unstable, switch --config to configs/100m-kaggle-fallback.yaml.
set -euo pipefail
cd /kaggle/working/Lyden
export PYTHONUNBUFFERED=1
for round in 1 2 3 4 5 6 7 8; do
  echo "[train round $round] start"
  if python scripts/train.py --config configs/100m-kaggle.yaml --val-every 600; then
    echo "[train round $round] completed"
    break
  else
    echo "[train round $round] exited nonzero; retrying from last checkpoint"
    sleep 5
  fi
done
ls -la /kaggle/working/Lyden/runs/100m/ckpts 2>/dev/null | tail -5 || echo '[train] no checkpoints yet'

In [ ]:
%%bash
set -uo pipefail
cd /kaggle/working/Lyden
CKPT=$(ls -1 runs/100m/ckpts/ckpt_*.pt 2>/dev/null | sort | tail -1)
if [ -z "$CKPT" ]; then echo '[eval] no checkpoints yet; skipping'; exit 0; fi
echo "[eval] $CKPT"
python scripts/eval.py \
    --config configs/100m-kaggle.yaml \
    --ckpt "$CKPT" \
    --tasks ppl,longctx,samples \
    --val-batches 40 \
    --lengths 512,1024 \
    --max-samples 3
echo '=== eval report ==='
cat runs/100m/eval_report.json | head -80